In [ ]:
import google.generativeai as genai

# Configure the API key
genai.configure(api_key='GOOGLE_API_KEY')


# List available models
print("Available Gemini Models:")
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model Name: {model.name}")
        print(f"Description: {model.description}")
        print(f"Input Token Limit: {model.input_token_limit}")
        print(f"Output Token Limit: {model.output_token_limit}")
        print("------")

# Initialize the model
model = genai.GenerativeModel('gemini-2.0-flash-001')


# Prompt Engineering Techniques

---

## 1. Zero-Shot Prompting

### Concept
Asking the model to perform a task without providing any examples, relying entirely on its pre-trained knowledge.

### Characteristics
- No examples provided
- Simple, direct instructions
- Relies on model's base training

### When to Use
- For straightforward classification tasks
- General knowledge questions
- Simple text transformations

### Example
```text
Classify this text as positive, neutral or negative:
"I loved the new movie, the acting was superb!"

In [ ]:
# Zero-shot prompt
prompt = """
Classify the following text as Terrorism, NonTerrorism, or Neutral:
"An explosion occurred near the city center causing multiple casualties and panic among civilians."
"""
response = model.generate_content(prompt)
print(response.text)

Neutral.

Here's why:

*   **Lack of Intent/Motivation:** The statement simply reports an event. It doesn't mention who caused the explosion, or what their motives might have been. Terrorism is defined by a political, religious, or ideological motive.
*   **Missing Context:** We don't know if the explosion was accidental (e.g., a gas leak), a criminal act (e.g., a bombing for revenge), or something else entirely. Without more information, it's impossible to classify it as terrorism.
*   **Neutral Language:** The language used is factual and descriptive, without any emotional or biased terms that might suggest terrorism.


## 2. Few-Shot Prompting  
### Concept  
Providing several input-output examples before the actual task.  This technique helps the model understand the desired pattern or format through demonstration.

### Characteristics  
- 3-5 demonstration examples  
- Shows desired format/pattern  
- Teaches through examples  

### When to Use  
- Complex formatting needs  
- Establishing response patterns  
- Niche domain tasks  

---

In [ ]:
few_shot_prompt = """
Text: "The armed group claimed responsibility for the bombing in the capital." Classification: Terrorism
Text: "Government officials launched a new education program in rural areas." Classification: NonTerrorism
Text: "The meeting was held to discuss regional development strategies." Classification: Neutral
Text: "A drone strike targeted suspected militant hideouts near the border." Classification: ?
"""

response = model.generate_content(few_shot_prompt)
print(response.text)

Classification: Terrorism



## 3. Chain-of-Thought Prompting  
### Concept  
Encouraging step-by-step reasoning explanations. You guide the model to generate intermediate reasoning steps rather than jumping directly to the final answer.

### Characteristics  
- Shows intermediate steps  
- "Let's think step by step" trigger  
- Transparent reasoning  

### When to Use  
- Math/logic problems  
- Complex decision making  
- Debugging scenarios  

---

In [ ]:
cot_prompt = """
Q: A news article reports: "A series of coordinated attacks were carried out on government buildings, resulting in numerous casualties." What is the classification of this text?
A: Let's think step by step.
"""

response = model.generate_content(cot_prompt)
print(response.text)

Okay, let's analyze the classification of the text "A series of coordinated attacks were carried out on government buildings, resulting in numerous casualties."

**1. Subject Matter:** The text describes violence directed at government buildings, causing harm to people. This points towards topics like:

*   **Politics:** Attacks on government buildings inherently involve political motivations or consequences.
*   **Crime/Violence:** The attacks are clearly criminal acts and involve violence.
*   **Conflict/Warfare/Terrorism:** Depending on the scale and intent, the attacks could be classified as acts of terrorism or even part of a larger conflict.
*   **Disaster/Emergency:** The "numerous casualties" suggest a disaster or emergency situation.

**2. Tone/Sentiment:** The text is factual and neutral in tone. It simply reports the events. The mention of "numerous casualties" evokes a sense of seriousness and potential tragedy.

**3. Type of Text:** This is clearly a news report or a factu

## 4. Role Prompting  
### Concept  
Assigning a specific persona/role to the model. You assign a role to the model at the beginning of the prompt to shape its response. This makes the model act as if it were a specific person, profession, or persona.



### Characteristics  
- Adopts professional tone  
- Provides contextual framing  
- Uses role qualifications  

### When to Use  
- Technical explanations  
- Professional communications  
- Simulated conversations  

---

In [ ]:
role_prompt = """
You are a terrorism analyst reviewing intelligence reports. Analyze the following text and classify it as Terrorism, NonTerrorism, or Neutral:

"A convoy of humanitarian aid trucks was attacked by unidentified gunmen in the northern province, leading to multiple fatalities."
"""

response = model.generate_content(role_prompt)
print(response.text)

Based on the provided text: "A convoy of humanitarian aid trucks was attacked by unidentified gunmen in the northern province, leading to multiple fatalities," the classification is **Likely Terrorism, but requires further investigation.**

Here's why:

* **Attack on a Convoy:** Attacking a humanitarian aid convoy is a significant action. Humanitarian organizations are generally considered neutral parties providing assistance to vulnerable populations. Targeting them suggests a motive beyond simple criminal activity.
* **Unidentified Gunmen:** The fact that the attackers are unidentified makes it difficult to immediately determine the motive. However, terrorist groups often operate anonymously or claim responsibility later to maximize the impact and spread fear.
* **Multiple Fatalities:** The presence of fatalities elevates the seriousness of the incident. Terrorist acts often aim to inflict maximum casualties to generate fear and instability.
* **Northern Province:** This geographic d

## 5. Template-Based Prompting  mainly used for rag
### Concept  
Using structured templates for output consistency.  Template-Based Prompting is a technique where you use structured, fill-in-the-blank style prompts to ensure consistency and control in how a language model responds. It's about creating reusable prompt formats with placeholders (like {input}, {question}, or {text}) that can be filled in dynamically.

### Characteristics  
- Enforces standardization  
- Automating responses at scale
- Field-based placeholders  
- Repeatable structure  

### When to Use  
- Product descriptions  
- Report generation  
- Batch content creation  

---

In [ ]:
# Define the prompt template
template = """
Classify the point of view of the following text {text} as Terrorism, NonTerrorism, or Neutral, and provide reasoning of the output.

Text: ""

Classification:
"""

# Example input text
input_text = "A suicide bombing targeted a crowded market, injuring dozens of civilians."

# Fill the template
prompt = template.format(text=input_text)

# Generate the response
response = model.generate_content(prompt)

# Output the response
print("Classification:", response.text.strip())

Classification: Text: A suicide bombing targeted a crowded market, injuring dozens of civilians.

Classification: Terrorism

Reasoning: The text describes an act of violence (suicide bombing) targeting civilians in a public space (crowded market). Suicide bombings are commonly associated with terrorist tactics, and the targeting of civilians suggests an intent to cause terror and disruption. Therefore, the text leans towards a "Terrorism" classification.


## 6. Instruction-Based Prompting  
### Concept  
Providing detailed, step-by-step task instructions.  

### Characteristics  
- Explicit requirements  
- Numbered steps  
- Clear success criteria  

### When to Use  
- Complex technical tasks  
- Multi-step operations  
- Precise output needs  

In [ ]:
instruction_prompt = """
Write a Python function that takes a list of numbers and returns a dictionary with the following structure:
{
    "sum": sum of all numbers,
    "average": average of numbers,
    "max": maximum value,
    "min": minimum value
}

Include type hints and a docstring explaining the function.
"""

response = model.generate_content(instruction_prompt)
print(response.text)

```python
from typing import List, Dict, Union


def analyze_numbers(numbers: List[Union[int, float]]) -> Dict[str, Union[int, float]]:
    """
    Analyzes a list of numbers and returns a dictionary containing the sum, average,
    maximum, and minimum values.

    Args:
        numbers: A list of numbers (integers or floats).

    Returns:
        A dictionary with the following keys:
        "sum": The sum of all numbers in the list.
        "average": The average of the numbers in the list.
        "max": The maximum value in the list.
        "min": The minimum value in the list.
        Returns an empty dictionary if the input list is empty.
    """

    if not numbers:
        return {}  # Handle empty list case

    total_sum = sum(numbers)
    average = total_sum / len(numbers)
    maximum = max(numbers)
    minimum = min(numbers)

    result = {
        "sum": total_sum,
        "average": average,
        "max": maximum,
        "min": minimum
    }
    return result


if __

## 7. Self-Consistency Prompting
### Concept  
Asking the model to verify or evaluate its own responses for accuracy.

### Characteristics  
- Self-validation mechanism  
- Error-checking built into prompt  
- Multiple reasoning paths  

### When to Use  
- Fact-checking responses  
- Improving answer reliability  
- Complex problem-solving  

In [ ]:
self_consistency_prompt = """
Q: What is 15% of 200?
A: 30
Verify if this answer is correct with multiple ways and explain your reasoning.
"""

response = model.generate_content(self_consistency_prompt)
print(response.text)

Okay, let's verify if 15% of 200 is indeed 30 using a few different methods:

**Method 1: Direct Calculation (Decimal Conversion)**

*   Convert the percentage to a decimal: 15% = 15/100 = 0.15
*   Multiply the decimal by the number: 0.15 * 200 = 30

**Method 2: Fraction Conversion**

*   Convert the percentage to a fraction: 15% = 15/100 = 3/20 (simplified)
*   Multiply the fraction by the number: (3/20) * 200 = 3 * (200/20) = 3 * 10 = 30

**Method 3: Breaking it Down**

*   10% of 200 is 20 (because 10/100 * 200 = 20)
*   5% of 200 is half of 10%, so it's 10 (because 5/100 * 200 = 10, or 20 / 2 = 10)
*   15% of 200 is 10% + 5%, so it's 20 + 10 = 30

**Method 4: Proportion**

*   Set up a proportion:  15/100 = x/200  (where x is the unknown value)
*   Cross-multiply: 100 * x = 15 * 200
*   Simplify: 100x = 3000
*   Divide both sides by 100: x = 30

**Conclusion**

All four methods consistently arrive at the answer of **30**.  Therefore, the statement "15% of 200 is 30" is **correct**.

In [ ]:
self_consistency_prompt = """
Q: What is 15% of 200?
A: 30
Verify if this answer is correct and explain your reasoning.
"""

response = model.generate_content(self_consistency_prompt)
print(response.text)

The answer is correct.

**Reasoning:**

To find 15% of 200, you can do the following:

*   **Convert the percentage to a decimal:** 15% = 15/100 = 0.15
*   **Multiply the decimal by the number:** 0.15 * 200 = 30

Therefore, 15% of 200 is indeed 30.



# Prompts with parameters

# Model Configuration Parameters

## Core Parameters

### Temperature
**Concept**: Controls randomness in output generation  
**Range**: 0.0 (deterministic) to 1.0 (creative)  
**Effects**:  
- Lower values → More focused, repetitive outputs  
- Higher values → More diverse, creative outputs  
**Recommended Use**:  
- 0.2-0.3: Factual Q&A  
- 0.5-0.7: Content generation  
- 0.8-1.0: Creative writing  

### Top-p (Nucleus Sampling)
**Concept**: Filters probability distribution of next tokens  
**Range**: 0.0 to 1.0  
**Effects**:  
- Lower values → More conservative word choices  
- Higher values → More diverse vocabulary  
**Recommended Use**:  
- 0.7-0.9 for most applications  
- Below 0.5 for highly constrained outputs  

### Max Output Tokens
**Concept**: Limits response length  
**Range**: Varies by model (typically 1-4096)  
**Effects**:  
- Too low → Truncated responses  
- Too high → Wasted computation  
**Recommended Use**:  
- 50-100: Single-sentence answers  
- 150-300: Paragraph responses  
- 500+: Long-form content  

## Advanced Parameters

### Top-k
**Concept**: Samples from top k most likely tokens  
**Range**: Positive integers (typically 1-100)  
**Interaction**: Works with temperature to control diversity  

### Frequency Penalty
**Concept**: Discourages repetitive phrases  
**Range**: 0.0 (no penalty) to 1.0 (strong penalty)  

### Presence Penalty
**Concept**: Encourages new topic introduction  
**Range**: 0.0 (no penalty) to 1.0 (strong penalty)  

## Configuration Examples

### Technical Documentation
```python
{
    "temperature": 0.3,
    "top_p": 0.7,
    "max_tokens": 500,
    "frequency_penalty": 0.5
}

In [ ]:
import google.generativeai as genai

def setup_gemini(api_key):
    """Configure Gemini API and list available models"""
    genai.configure(api_key=api_key)

    print("\nAvailable Models:")
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name} (Input: {m.input_token_limit}, Output: {m.output_token_limit})")

def generate_with_parameters(prompt, temperature=0.7, top_p=0.9, max_tokens=200):
    """Generate content with specific parameters"""
    model = genai.GenerativeModel(
        'gemini-2.0-flash',
        generation_config={
            "temperature": temperature,
            "top_p": top_p,
            "max_output_tokens": max_tokens
        }
    )

    response = model.generate_content(prompt)
    return response.text

# Example usage
if __name__ == "__main__":
    API_KEY = "GOOGLE_API_KEY"  

    # Setup and list models
    setup_gemini(API_KEY)

    # Example 1: Zero-shot prompting with default parameters
    prompt1 = """
    Classify the following text as positive, neutral, or negative:
    "The new update made the app much more intuitive to use!"
    """
    print("\nZero-shot example (default params):")
    print(generate_with_parameters(prompt1))

    # Example 2: Few-shot prompting with creative parameters
    prompt2 = """
    Text: "This product broke after 2 days of use." Sentiment: negative
    Text: "The service was okay, nothing special." Sentiment: neutral
    Text: "I absolutely love this new feature!" Sentiment: positive
    Text: "The delivery was faster than expected." Sentiment: ?
    """
    print("\nFew-shot example (creative params - temperature=0.9):")
    print(generate_with_parameters(prompt2, temperature=0.9))

    # Example 3: Chain-of-thought with precise parameters
    prompt3 = """
    Q: A bookstore has 120 books. They sold 30 books on Monday and 45 books on Tuesday.
    How many books are left?
    A: Let's think step by step.
    """
    print("\nChain-of-thought example (precise params - temperature=0.3):")
    print(generate_with_parameters(prompt3, temperature=0.3))

    # Example 4: Role prompting with longer response
    prompt4 = """
    You are a senior software engineer reviewing code. Analyze the following Python function
    and suggest improvements:

    def calculate_average(numbers):
        return sum(numbers)/len(numbers)
    """
    print("\nRole prompting example (longer response - max_tokens=400):")
    print(generate_with_parameters(prompt4, max_tokens=400))